# Customer Project Overview

The telecom operator Interconnect would like to be able to forecast their churn of clients. If it's discovered that a user is planning to leave, they will be offered promotional codes and special plan options. Interconnect's marketing team has collected some of their clientele's personal data, including information about their plans and contracts.

Interconnect's services Interconnect mainly provides two types of services:

Landline communication. The telephone can be connected to several lines simultaneously. Internet. The network can be set up via a telephone line (DSL, digital subscriber line) or through a fiber optic cable. Some other services the company provides include:

Internet security: antivirus software (DeviceProtection) and a malicious website blocker (OnlineSecurity) A dedicated technical support line (TechSupport) Cloud file storage and data backup (OnlineBackup) TV streaming (StreamingTV) and a movie directory (StreamingMovies) The clients can choose either a monthly payment or sign a 1- or 2-year contract. They can use various payment methods and receive an electronic invoice after a transaction.

## Initializing project

In [1]:
import numpy as np
import pandas as pd
import time as time
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings(
    "ignore",
    message="No further splits with positive gain"
)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import make_scorer, roc_auc_score, classification_report, confusion_matrix

## Loading the data

In [2]:
df_contract = pd.read_csv('datasets/contract.csv')
df_personal = pd.read_csv('datasets/personal.csv')
df_internet = pd.read_csv('datasets/internet.csv')
df_phone = pd.read_csv('datasets/phone.csv')

## Checking the data

In [3]:
display(df_contract)
display(df_contract.info())
display(df_contract.shape)
display(df_contract.isnull().sum())
display(df_contract['EndDate'].value_counts().get('No', 0))

,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65
...,...,...,...,...,...,...,...,...
7038,6840-RESVB,2018-02-01,No,One year,Yes,Mailed check,84.80,1990.5
7039,2234-XADUH,2014-02-01,No,One year,Yes,Credit card (automatic),103.20,7362.9
7040,4801-JZAZL,2019-03-01,No,Month-to-month,Yes,Electronic check,29.60,346.45
7041,8361-LTMKD,2019-07-01,2019-11-01 00:00:00,Month-to-month,Yes,Mailed check,74.40,306.6


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
dtypes: float64(1), object(7)
memory usage: 440.3+ KB


None

(7043, 8)

customerID          0
BeginDate           0
EndDate             0
Type                0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
dtype: int64

np.int64(5174)

In [4]:
display(df_personal)
display(df_personal.info())
display(df_personal.shape)
display(df_personal.isnull().sum())

,customerID,gender,SeniorCitizen,Partner,Dependents
0,7590-VHVEG,Female,0,Yes,No
1,5575-GNVDE,Male,0,No,No
2,3668-QPYBK,Male,0,No,No
3,7795-CFOCW,Male,0,No,No
4,9237-HQITU,Female,0,No,No
...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes
7039,2234-XADUH,Female,0,Yes,Yes
7040,4801-JZAZL,Female,0,Yes,Yes
7041,8361-LTMKD,Male,1,Yes,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     7043 non-null   object
 1   gender         7043 non-null   object
 2   SeniorCitizen  7043 non-null   int64 
 3   Partner        7043 non-null   object
 4   Dependents     7043 non-null   object
dtypes: int64(1), object(4)
memory usage: 275.2+ KB


None

(7043, 5)

customerID       0
gender           0
SeniorCitizen    0
Partner          0
Dependents       0
dtype: int64

In [5]:
display(df_internet)
display(df_internet.info())
display(df_internet.shape)
display(df_internet.isnull().sum())

,customerID,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,7590-VHVEG,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Fiber optic,No,No,No,No,No,No
...,...,...,...,...,...,...,...,...
5512,6840-RESVB,DSL,Yes,No,Yes,Yes,Yes,Yes
5513,2234-XADUH,Fiber optic,No,Yes,Yes,No,Yes,Yes
5514,4801-JZAZL,DSL,Yes,No,No,No,No,No
5515,8361-LTMKD,Fiber optic,No,No,No,No,No,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5517 entries, 0 to 5516
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerID        5517 non-null   object
 1   InternetService   5517 non-null   object
 2   OnlineSecurity    5517 non-null   object
 3   OnlineBackup      5517 non-null   object
 4   DeviceProtection  5517 non-null   object
 5   TechSupport       5517 non-null   object
 6   StreamingTV       5517 non-null   object
 7   StreamingMovies   5517 non-null   object
dtypes: object(8)
memory usage: 344.9+ KB


None

(5517, 8)

customerID          0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
dtype: int64

In [6]:
display(df_phone)
display(df_phone.info())
display(df_phone.shape)
display(df_phone.isnull().sum())

,customerID,MultipleLines
0,5575-GNVDE,No
1,3668-QPYBK,No
2,9237-HQITU,No
3,9305-CDSKC,Yes
4,1452-KIOVK,Yes
...,...,...
6356,2569-WGERO,No
6357,6840-RESVB,Yes
6358,2234-XADUH,Yes
6359,8361-LTMKD,Yes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6361 entries, 0 to 6360
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     6361 non-null   object
 1   MultipleLines  6361 non-null   object
dtypes: object(2)
memory usage: 99.5+ KB


None

(6361, 2)

customerID       0
MultipleLines    0
dtype: int64

In [7]:
# Checking for duplicates
display(df_contract[df_contract.duplicated()])
display(df_personal[df_personal.duplicated()])
display(df_internet[df_internet.duplicated()])
display(df_phone[df_phone.duplicated()])

,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges


,customerID,gender,SeniorCitizen,Partner,Dependents


,customerID,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies


,customerID,MultipleLines


The data frames appear to be clean of dupes and null values at this point. I may need to merge some of the data together in future analysis.

## Questions and Work Plan

- Contracts and Personal have the same rows. I didn't see any dupes or null values. Are they one-to-one? Can I cleanly merge them with no issues?
- Phones have a different number of rows from contracts and Personal. Shouldn't they have the same number of rows since they are saved under the same customerID key?
- For the Internet dataframe, I don't see a costumerID field, I see a user_id field. How do I determine how to attribute the data to a customerID? How will I be able to merge the data into something usable?
- In order to forecast churn from the current data set, I will need to exclude all the 'No' values from "EndDate" column. This leaves fewer than 2000 entries; this means the training and validation set will be about less than 1/7 of the total data set, and the test set will be huge relative to that. Will this be acceptable to management, considering it may produce a larger variance?

Steps:
- Step 1: Data Cleaning and Preprocessing. This will include merging the data sets into a more complete dataframe.
- Step 2: Feature engineering and selection.
- Step 3: Splitting of training, validation, and testing data sets.
- Step 4: Model selection, optimization, and training.
- Step 5: Final model selection and testing.

## Data prep and exploritory analysis

In [8]:
def standardize_column_names(df):
    df.columns = (
        df.columns
        .str.strip()                      # Remove leading/trailing whitespace
        .str.lower()                     # Convert to lowercase
        .str.replace(' ', '_')           # Replace spaces with underscores
        .str.replace(r'[^\w]', '', regex=True)  # Remove special characters
    )
    return df

In [9]:
# Cleaning up column names
standardize_column_names(df_contract)
standardize_column_names(df_personal)
standardize_column_names(df_internet)
standardize_column_names(df_phone)

display(df_contract)
display(df_personal)
display(df_internet)
display(df_phone)

,customerid,begindate,enddate,type,paperlessbilling,paymentmethod,monthlycharges,totalcharges
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65
...,...,...,...,...,...,...,...,...
7038,6840-RESVB,2018-02-01,No,One year,Yes,Mailed check,84.80,1990.5
7039,2234-XADUH,2014-02-01,No,One year,Yes,Credit card (automatic),103.20,7362.9
7040,4801-JZAZL,2019-03-01,No,Month-to-month,Yes,Electronic check,29.60,346.45
7041,8361-LTMKD,2019-07-01,2019-11-01 00:00:00,Month-to-month,Yes,Mailed check,74.40,306.6


,customerid,gender,seniorcitizen,partner,dependents
0,7590-VHVEG,Female,0,Yes,No
1,5575-GNVDE,Male,0,No,No
2,3668-QPYBK,Male,0,No,No
3,7795-CFOCW,Male,0,No,No
4,9237-HQITU,Female,0,No,No
...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes
7039,2234-XADUH,Female,0,Yes,Yes
7040,4801-JZAZL,Female,0,Yes,Yes
7041,8361-LTMKD,Male,1,Yes,No


,customerid,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies
0,7590-VHVEG,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Fiber optic,No,No,No,No,No,No
...,...,...,...,...,...,...,...,...
5512,6840-RESVB,DSL,Yes,No,Yes,Yes,Yes,Yes
5513,2234-XADUH,Fiber optic,No,Yes,Yes,No,Yes,Yes
5514,4801-JZAZL,DSL,Yes,No,No,No,No,No
5515,8361-LTMKD,Fiber optic,No,No,No,No,No,No


,customerid,multiplelines
0,5575-GNVDE,No
1,3668-QPYBK,No
2,9237-HQITU,No
3,9305-CDSKC,Yes
4,1452-KIOVK,Yes
...,...,...
6356,2569-WGERO,No
6357,6840-RESVB,Yes
6358,2234-XADUH,Yes
6359,8361-LTMKD,Yes


In [10]:
# Checking duplicates
duplicates = df_contract[df_contract['customerid'].duplicated()]
display(duplicates)

duplicates2 = df_personal[df_personal['customerid'].duplicated()]
display(duplicates2)

,customerid,begindate,enddate,type,paperlessbilling,paymentmethod,monthlycharges,totalcharges


,customerid,gender,seniorcitizen,partner,dependents


In [11]:
# Checking dataframes for duplicate customerid
df_complete = pd.merge(df_contract, df_personal, on='customerid')
display(df_complete)

,customerid,begindate,enddate,type,paperlessbilling,paymentmethod,monthlycharges,totalcharges,gender,seniorcitizen,partner,dependents
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85,Female,0,Yes,No
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5,Male,0,No,No
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15,Male,0,No,No
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75,Male,0,No,No
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65,Female,0,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,2018-02-01,No,One year,Yes,Mailed check,84.80,1990.5,Male,0,Yes,Yes
7039,2234-XADUH,2014-02-01,No,One year,Yes,Credit card (automatic),103.20,7362.9,Female,0,Yes,Yes
7040,4801-JZAZL,2019-03-01,No,Month-to-month,Yes,Electronic check,29.60,346.45,Female,0,Yes,Yes
7041,8361-LTMKD,2019-07-01,2019-11-01 00:00:00,Month-to-month,Yes,Mailed check,74.40,306.6,Male,1,Yes,No


In [12]:
# Merging the df_complete with df_phone
df_phone['phoneservice'] = 'Yes'
df_complete = pd.merge(df_complete, df_phone, on='customerid', how='left')
display(df_complete)

,customerid,begindate,enddate,type,paperlessbilling,paymentmethod,monthlycharges,totalcharges,gender,seniorcitizen,partner,dependents,multiplelines,phoneservice
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85,Female,0,Yes,No,NaN,NaN
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5,Male,0,No,No,No,Yes
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15,Male,0,No,No,No,Yes
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75,Male,0,No,No,NaN,NaN
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65,Female,0,No,No,No,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,2018-02-01,No,One year,Yes,Mailed check,84.80,1990.5,Male,0,Yes,Yes,Yes,Yes
7039,2234-XADUH,2014-02-01,No,One year,Yes,Credit card (automatic),103.20,7362.9,Female,0,Yes,Yes,Yes,Yes
7040,4801-JZAZL,2019-03-01,No,Month-to-month,Yes,Electronic check,29.60,346.45,Female,0,Yes,Yes,NaN,NaN
7041,8361-LTMKD,2019-07-01,2019-11-01 00:00:00,Month-to-month,Yes,Mailed check,74.40,306.6,Male,1,Yes,No,Yes,Yes


In [13]:
# Merging the df_complete with df_internet
df_complete = pd.merge(df_complete, df_internet, on='customerid', how='left')
display(df_complete)

,customerid,begindate,enddate,type,paperlessbilling,paymentmethod,monthlycharges,totalcharges,gender,seniorcitizen,...,dependents,multiplelines,phoneservice,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85,Female,0,...,No,NaN,NaN,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5,Male,0,...,No,No,Yes,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15,Male,0,...,No,No,Yes,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75,Male,0,...,No,NaN,NaN,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65,Female,0,...,No,No,Yes,Fiber optic,No,No,No,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,2018-02-01,No,One year,Yes,Mailed check,84.80,1990.5,Male,0,...,Yes,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes
7039,2234-XADUH,2014-02-01,No,One year,Yes,Credit card (automatic),103.20,7362.9,Female,0,...,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes
7040,4801-JZAZL,2019-03-01,No,Month-to-month,Yes,Electronic check,29.60,346.45,Female,0,...,Yes,NaN,NaN,DSL,Yes,No,No,No,No,No
7041,8361-LTMKD,2019-07-01,2019-11-01 00:00:00,Month-to-month,Yes,Mailed check,74.40,306.6,Male,1,...,No,Yes,Yes,Fiber optic,No,No,No,No,No,No


In [14]:
# Replacing null values with unknown.
df_complete.fillna('Unknown', inplace=True)
print(df_complete.isnull().sum())

customerid          0
begindate           0
enddate             0
type                0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
gender              0
seniorcitizen       0
partner             0
dependents          0
multiplelines       0
phoneservice        0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
dtype: int64


In [15]:
# Identifying non-churned from churn in a numerical value.
df_complete['churned'] = (
    df_complete['enddate']
      .map({'No': 0})
      .fillna(1)
      .astype(int)
)
display(df_complete.head())

,customerid,begindate,enddate,type,paperlessbilling,paymentmethod,monthlycharges,totalcharges,gender,seniorcitizen,...,multiplelines,phoneservice,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,churned
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85,Female,0,...,Unknown,Unknown,DSL,No,Yes,No,No,No,No,0
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5,Male,0,...,No,Yes,DSL,Yes,No,Yes,No,No,No,0
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15,Male,0,...,No,Yes,DSL,Yes,Yes,No,No,No,No,1
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75,Male,0,...,Unknown,Unknown,DSL,Yes,No,Yes,Yes,No,No,0
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65,Female,0,...,No,Yes,Fiber optic,No,No,No,No,No,No,1


In [16]:
# Converting totalcharges to float.
df_complete['totalcharges'] = pd.to_numeric(df_complete['totalcharges'], errors='coerce')

# Dropping nan values
df_complete = df_complete.dropna(subset=['totalcharges']).reset_index(drop=True)
# Checking the work.
print("NaN values in totalcharges after conversion:")
print(df_complete['totalcharges'].isna().sum())

NaN values in totalcharges after conversion:
0


## Modeling

In [17]:
# Splitting into training, validation, and test sets.

# First split into train+val and test (80/20)
train_val, test = train_test_split(
    df_complete, 
    test_size=0.2, 
    stratify=df_complete['churned'], 
    random_state=12345
)

# Then split train+val into train and val (75/25 of 80% → 60/20)
train, valid = train_test_split(
    train_val, 
    test_size=0.25, 
    stratify=train_val['churned'], 
    random_state=12345
)

In [18]:
# Confirming churn ratio
print("Train churn ratio:")
print(train['churned'].value_counts(normalize=True))

print("\nValidation churn ratio:")
print(valid['churned'].value_counts(normalize=True))

print("\nTest churn ratio:")
print(test['churned'].value_counts(normalize=True))

Train churn ratio:
churned
0    0.734234
1    0.265766
Name: proportion, dtype: float64

Validation churn ratio:
churned
0    0.734186
1    0.265814
Name: proportion, dtype: float64

Test churn ratio:
churned
0    0.734186
1    0.265814
Name: proportion, dtype: float64


In [19]:
# Split into features and target

features_train = train.drop('churned', axis=1)
target_train = train['churned']

features_valid = valid.drop('churned', axis=1)
target_valid = valid['churned']

features_test = test.drop('churned', axis=1)
target_test = test['churned']


In [20]:
# Encoding columns
cat_cols= ['type', 'paymentmethod', 'gender','internetservice']
binary_cols = [
    'paperlessbilling','partner','dependents',
    'multiplelines','onlinesecurity','onlinebackup',
    'deviceprotection','techsupport','streamingtv','streamingmovies','phoneservice'
]

In [21]:
# Encoding Function

def encode_set(df, binary_cols, cat_cols, yes_no_map=None, drop_first=True, reference_cols=None):
    yes_no_map = yes_no_map or {'Yes': 1, 'No': 0, 'Unknown': 0}

    # work on a copy
    df_enc = df.copy()
    
    # 1) map binaries
    df_enc[binary_cols] = (
        df_enc[binary_cols]
            .replace(yes_no_map)
            .infer_objects(copy=False)
    )

    # 2) one-hot encode
    df_enc = pd.get_dummies(
        df_enc,
        columns=cat_cols,
        drop_first=drop_first
    )

    # 3) ensure consistent columns (useful when encoding val/test)
    if reference_cols is not None:
        df_enc = df_enc.reindex(columns=reference_cols, fill_value=0)

    return df_enc


In [22]:
# Fixing values
# Encode each of the feature sets
features_enc_test = encode_set(features_test, binary_cols, cat_cols)
features_enc_valid = encode_set(features_valid, binary_cols, cat_cols)
features_enc_train = encode_set(features_train, binary_cols, cat_cols)

# Remove ID and date columns
columns_to_drop = ['customerid', 'begindate', 'enddate']
features_enc_train = features_enc_train.drop(columns=columns_to_drop)
features_enc_valid = features_enc_valid.drop(columns=columns_to_drop)
features_enc_test = features_enc_test.drop(columns=columns_to_drop)

# Converting totalcharges to a float.
# Use pd.to_numeric with errors='coerce' to handle non-numeric values
features_enc_train['totalcharges'] = pd.to_numeric(features_enc_train['totalcharges'], errors='coerce')
features_enc_valid['totalcharges'] = pd.to_numeric(features_enc_valid['totalcharges'], errors='coerce')
features_enc_test['totalcharges'] = pd.to_numeric(features_enc_test['totalcharges'], errors='coerce')

# Fill NaN values with 0 (meaning "No" for internet services)
features_enc_train = features_enc_train.fillna(0)
features_enc_valid = features_enc_valid.fillna(0)
features_enc_test = features_enc_test.fillna(0)

# Verify no more NaN values
print("NaN values remaining:")
print(features_enc_train.isna().sum().sum())

NaN values remaining:
0


C:\Users\mytho\AppData\Local\Temp\ipykernel_15940\3512633656.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(yes_no_map)
C:\Users\mytho\AppData\Local\Temp\ipykernel_15940\3512633656.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(yes_no_map)
C:\Users\mytho\AppData\Local\Temp\ipykernel_15940\3512633656.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to t

In [23]:
# Checking encoding
display(features_enc_test.head())
display(features_enc_valid.head())
display(features_enc_train.head())

,paperlessbilling,monthlycharges,totalcharges,seniorcitizen,partner,dependents,multiplelines,phoneservice,onlinesecurity,onlinebackup,...,streamingtv,streamingmovies,type_One year,type_Two year,paymentmethod_Credit card (automatic),paymentmethod_Electronic check,paymentmethod_Mailed check,gender_Male,internetservice_Fiber optic,internetservice_Unknown
2507,1,68.75,1689.45,0,1,0,0,1,1,0,...,0,1,True,False,False,False,False,False,False,False
3446,1,44.95,44.95,0,0,0,0,1,0,0,...,0,0,False,False,False,False,True,True,False,False
6808,1,98.80,5617.75,1,0,0,1,1,1,0,...,1,1,False,False,False,True,False,True,True,False
4430,1,24.95,1222.25,0,1,1,1,1,0,0,...,0,0,False,True,False,False,True,True,False,True
2546,0,19.45,227.45,0,0,0,0,1,0,0,...,0,0,False,False,False,False,False,False,False,True


,paperlessbilling,monthlycharges,totalcharges,seniorcitizen,partner,dependents,multiplelines,phoneservice,onlinesecurity,onlinebackup,...,streamingtv,streamingmovies,type_One year,type_Two year,paymentmethod_Credit card (automatic),paymentmethod_Electronic check,paymentmethod_Mailed check,gender_Male,internetservice_Fiber optic,internetservice_Unknown
441,0,101.55,5070.40,1,0,0,1,1,1,0,...,1,1,False,False,True,False,False,False,True,False
3186,1,29.35,381.20,1,0,0,0,0,0,1,...,0,0,False,False,True,False,False,True,False,False
4481,1,53.85,3399.85,0,1,1,1,1,0,1,...,0,0,True,False,False,False,False,False,False,False
5249,1,104.70,7220.35,0,0,0,1,1,0,1,...,1,1,True,False,False,False,False,True,True,False
1752,1,79.95,4362.05,0,1,1,0,1,0,1,...,0,0,False,True,False,False,False,True,True,False


,paperlessbilling,monthlycharges,totalcharges,seniorcitizen,partner,dependents,multiplelines,phoneservice,onlinesecurity,onlinebackup,...,streamingtv,streamingmovies,type_One year,type_Two year,paymentmethod_Credit card (automatic),paymentmethod_Electronic check,paymentmethod_Mailed check,gender_Male,internetservice_Fiber optic,internetservice_Unknown
4433,0,20.25,835.50,0,1,0,0,1,0,0,...,0,0,False,True,False,True,False,True,False,True
905,1,65.35,1231.85,0,0,0,0,1,0,1,...,0,1,True,False,False,False,False,True,False,False
2168,0,86.05,308.10,0,0,0,0,1,0,0,...,0,1,False,False,False,True,False,True,True,False
2189,1,79.50,79.50,0,0,0,0,1,0,0,...,1,0,False,False,False,True,False,True,True,False
4514,0,20.70,587.10,0,1,0,0,1,0,0,...,0,0,True,False,True,False,False,True,False,True


In [24]:
# Validate Model Function
def validate_model(model, features_val, target_val):
    y_pred = model.predict(features_val)
    y_proba = model.predict_proba(features_val)[:, 1]
    
    auc = roc_auc_score(target_val, y_proba)
    print(f"Additional validation metrics:")
    print(f"AUC-ROC: {auc:.4f}")
    
# Model Evaluation Function
def tune_model_auc(model, param_grid, features_train, target_train, cv=6, features_val=None, target_val=None, validate=True):
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        scoring='roc_auc',
        cv=StratifiedKFold(n_splits=cv),
        n_iter=125,
        n_jobs=-1,
        verbose=1,
        random_state=42
    )

    search.fit(features_train, target_train)

    print(f"Best AUC (CV): {search.best_score_:.4f}")
    print(f"Best Params: {search.best_params_}")

    best_model = search.best_estimator_

    if validate and features_val is not None and target_val is not None:
        print("\nValidating on validation set...")
        val_auc = roc_auc_score(target_val, best_model.predict_proba(features_val)[:, 1])
        print(f"Validation AUC-ROC: {val_auc:.4f}")
        validate_model(best_model, features_val, target_val)

    return best_model

In [25]:
# Logistic regression as a benchmark.

# Initialize model
logreg_model = LogisticRegression(
    penalty='l2',           # Regularization
    solver='liblinear',     # Good for small datasets
    class_weight='balanced' # Handles class imbalance
)

# Fit and predict
logreg_model.fit(features_enc_train, target_train)
y_pred = logreg_model.predict(features_enc_valid)
y_proba = logreg_model.predict_proba(features_enc_valid)[:, 1]

# Evaluate
auc = roc_auc_score(target_valid, y_proba)
print(f"Baseline AUC-ROC: {auc:.4f}")



Baseline AUC-ROC: 0.8407


In [26]:
# Decision tree
dt_model = DecisionTreeClassifier()
dt_params = {
    'max_depth': [None, 5, 10, 20, 30],       # More control over complexity
    'min_samples_split': [2, 10, 20, 50],     # Regularization
    'min_samples_leaf': [1, 2, 5, 10, 20],    # Smooth predictions
    'max_features': [None, 'sqrt', 'log2'],   # Feature selection
    'class_weight': ['balanced']
}

best_dt = tune_model_auc(dt_model, dt_params, features_enc_train, target_train, features_val = features_enc_valid, target_val=target_valid)

Fitting 6 folds for each of 125 candidates, totalling 750 fits
Best AUC (CV): 0.8159
Best Params: {'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 5, 'class_weight': 'balanced'}

Validating on validation set...
Validation AUC-ROC: 0.8253
Additional validation metrics:
AUC-ROC: 0.8253


In [27]:
# Random forest
rf_model = RandomForestClassifier()
rf_params = {
    'n_estimators': [100, 300, 500],  # More trees = better averaging
    'max_depth': [None, 10, 30, 50],        # Explore deeper trees
    'min_samples_split': [2, 5, 10, 20],    # Regularization
    'min_samples_leaf': [1, 2, 5, 10],      # Prevent overfitting
    'max_features': ['sqrt', 'log2', None], # Feature sampling
    'bootstrap': [True, False],             # Try both sampling strategies
    'class_weight': ['balanced']
}

best_rf = tune_model_auc(rf_model, rf_params, features_enc_train, target_train,features_val = features_enc_valid, target_val=target_valid)

Fitting 6 folds for each of 125 candidates, totalling 750 fits
Best AUC (CV): 0.8411
Best Params: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': 'balanced', 'bootstrap': True}

Validating on validation set...
Validation AUC-ROC: 0.8515
Additional validation metrics:
AUC-ROC: 0.8515


In [ ]:
# Light gradient boost
lgb_model = LGBMClassifier()
lgb_params = {
    'n_estimators': [300, 600, 1000],  # More granularity
    'learning_rate': [0.01, 0.05, 0.1],  # Finer control
    'max_depth': [5, 10, 15, -1],  # Explore deeper trees
    'num_leaves': [31, 63, 127],  # More expressive splits
    'min_child_samples': [5, 10, 20, 80],  # Regularization
    'reg_alpha': [0, 0.1, 0.5],  # L1 regularization
    'reg_lambda': [1, 5, 10],  # L2 regularization
    'subsample': [0.6, 0.8, 1.0],  # Row sampling
    'colsample_bytree': [0.6, 0.8, 1.0],  # Feature sampling
    'class_weight': ['balanced']
}

best_lgb = tune_model_auc(lgb_model, lgb_params, features_enc_train, target_train,features_val = features_enc_valid, target_val=target_valid)

Fitting 6 folds for each of 125 candidates, totalling 750 fits


In [ ]:
# Removed XGBoost since it takes too much runtime and doesn't complete before crashing.
'''# XGBoost
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'scale_pos_weight': [1, 5],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1, 10]
}

best_xgb = tune_model_auc(xgb_model, xgb_params, features_enc_train, target_train, cv=5)'''

My current best model is the LGB model with a AUC of 0.8423. I would like to see if I can get it any higher. I will look into feature engineering and then try to remodel. I am shooting for the 0.88 for the full 6 SP.

## Looking into feature importance and exploring

In [ ]:
# Function for checking for feature importance.
def plot_feature_importance(model, feature_names, top_n=20):
    importances = model.feature_importances_
    feat_imp_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False).head(top_n)

    plt.figure(figsize=(10, 6))
    plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='skyblue')
    plt.gca().invert_yaxis()
    plt.title("Top Feature Importances")
    plt.xlabel("Importance Score")
    plt.tight_layout()
    plt.show()

In [ ]:
# Feature importance for Decision Tree
plot_feature_importance(best_dt, features_enc_train.columns)

In [ ]:
# Feature importance for Random Forest
plot_feature_importance(best_rf, features_enc_train.columns)

In [ ]:
# Feature importance for LGB
plot_feature_importance(best_lgb, features_enc_train.columns)

Based on the feature importance, I am noticing a few key takeaways.
1. Type looks to play a big part. This makes sense since contract length will definitely affect churn.
2. Totalcharges and monthlycharges both appear highly ranked. Higher charges often will lead to churn.
3. Internetservice_fiberoptic shows up. Will further investigate if this is a pricing or satisfaction issue.
4. paymenthmethod_electronoiccheck. The idea of payment friction maybe correlated to churn.
I will feature engineer further to try to increase the AUC score.

## Feature engineering and reruning models

In [ ]:
df_rerun = df_complete.copy()
print(df_complete.head())
print(df_rerun.head())

In [ ]:
# Average monthly spend (tenure-adjusted)
df_rerun['avg_monthly_spend'] = df_rerun['totalcharges'] / df_rerun['monthlycharges']

In [ ]:
# Price per service ratio
# Counting up total services.
service_cols = [
    'phoneservice',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'multiplelines'
]
df_rerun['totalservices'] = df_rerun[service_cols].apply(
    lambda row: sum(val not in ['No', 'Unknown', None, np.nan] for val in row),
    axis=1
)
df_rerun['charges_per_service'] = df_rerun['monthlycharges'] / df_rerun['totalservices']

In [ ]:
# Adding a duration feature
# Ensure enddate column is treated as string first
df_rerun['enddate'] = df_rerun['enddate'].astype(str)

# Replace 'No' with the last known date
df_rerun['enddate'] = df_rerun['enddate'].replace('No', '2020-01-31')

# Strip any trailing time component (e.g. " 00:00:00") before parsing
df_rerun['enddate'] = df_rerun['enddate'].str.split(' ').str[0]

# Convert the column to datetime format
df_rerun['enddate'] = pd.to_datetime(df_rerun['enddate'], format='%Y-%m-%d')
df_rerun['begindate'] = pd.to_datetime(df_rerun['begindate'], format='%Y-%m-%d')

df_rerun['duration'] = ((df_rerun['enddate'] - df_rerun['begindate']).dt.days / 30.4375).round().astype(int)

# Contract value density
df_rerun['contract_value_density'] = df_rerun['totalcharges'] / df_rerun['duration']

In [ ]:
# Encoding the dataframe
df_enc2 = encode_set(df_rerun, binary_cols, cat_cols)
# High-risk payment + service combo
df_enc2['risky_combo'] = (df_enc2['paymentmethod_Electronic check'] == 1) & (df_enc2['internetservice_Fiber optic'] == 1)

In [ ]:
print(df_enc2.head())
print(df_enc2.shape)
print(df_enc2.isnull().sum())

In [ ]:
train_val2, test2 = train_test_split(
    df_enc2, 
    test_size=0.2, 
    stratify=df_enc2['churned'], 
    random_state=42
)

# Then split train+val into train and val (75/25 of 80% → 60/20)
train2, valid2 = train_test_split(
    train_val2, 
    test_size=0.25, 
    stratify=train_val2['churned'], 
    random_state=42
)

In [ ]:
features_train2 = train2.drop('churned', axis=1)
target_train2 = train2['churned']

features_valid2 = valid2.drop('churned', axis=1)
target_valid2 = valid2['churned']

features_test2 = test2.drop('churned', axis=1)
target_test2 = test2['churned']

# Remove ID and date columns
columns_to_drop = ['customerid', 'begindate', 'enddate']
features_train2 = features_train2.drop(columns=columns_to_drop)
features_valid2 = features_valid2.drop(columns=columns_to_drop)
features_test2 = features_test2.drop(columns=columns_to_drop)

In [ ]:
# Logistic regression as a benchmark.
# Initialize model
logreg_model2 = LogisticRegression(
    penalty='l2',           # Regularization
    solver='liblinear',     # Good for small datasets
    class_weight='balanced' # Handles class imbalance
)
# Fit and predict
logreg_model2.fit(features_train2, target_train2)
y_pred2 = logreg_model2.predict(features_valid2)
y_proba2 = logreg_model2.predict_proba(features_valid2)[:, 1]

# Evaluate
auc = roc_auc_score(target_valid2, y_proba2)
print(f"Baseline AUC-ROC: {auc:.4f}")

In [ ]:
# Decision tree
dt_model2 = DecisionTreeClassifier()
dt_params = {
    'max_depth': [None, 5, 10, 20, 30],       # More control over complexity
    'min_samples_split': [2, 10, 20, 50],     # Regularization
    'min_samples_leaf': [1, 2, 5, 10, 20],    # Smooth predictions
    'max_features': [None, 'sqrt', 'log2'],   # Feature selection
    'class_weight': ['balanced']
}

best_dt2 = tune_model_auc(dt_model2, dt_params, features_train2, target_train2, features_val = features_valid2, target_val=target_valid2)

In [ ]:
# Random forest
rf_model2 = RandomForestClassifier()
rf_params = {
    'n_estimators': [100, 300, 500],  # More trees = better averaging
    'max_depth': [None, 10, 30, 50],        # Explore deeper trees
    'min_samples_split': [2, 5, 10, 20],    # Regularization
    'min_samples_leaf': [1, 2, 5, 10],      # Prevent overfitting
    'max_features': ['sqrt', 'log2', None], # Feature sampling
    'bootstrap': [True, False],             # Try both sampling strategies
    'class_weight': ['balanced']
}

best_rf2 = tune_model_auc(rf_model2, rf_params, features_train2, target_train2, features_val = features_valid2, target_val=target_valid2)

In [ ]:
# Light gradient boost
lgb_model2 = LGBMClassifier()
lgb_params = {
    'n_estimators': [300, 600, 1000],  # More granularity
    'learning_rate': [0.01, 0.05, 0.1],  # Finer control
    'max_depth': [5, 10, 15, -1],  # Explore deeper trees
    'num_leaves': [31, 63, 127],  # More expressive splits
    'min_child_samples': [5, 10, 20, 80],  # Regularization
    'reg_alpha': [0, 0.1, 0.5],  # L1 regularization
    'reg_lambda': [1, 5, 10],  # L2 regularization
    'subsample': [0.6, 0.8, 1.0],  # Row sampling
    'colsample_bytree': [0.6, 0.8, 1.0],  # Feature sampling
    'class_weight': ['balanced']
}

best_lgb2 = tune_model_auc(lgb_model2, lgb_params, features_train2, target_train2, features_val = features_valid2, target_val=target_valid2)

The feature engineering has returned AUC-ROC scores that were less accurate than my original LGBModel. I will be using the original one without feature engineering for my final test to confirm the results.

## Final model test

In [ ]:
# Final test with best model
# Predict probabilities and labels
y_test_proba = best_lgb.predict_proba(features_enc_test)[:, 1]
y_test_pred = best_lgb.predict(features_enc_test)

# AUC-ROC Score
test_auc = roc_auc_score(target_test, y_test_proba)
print(f"🧪 Final Test AUC-ROC: {test_auc:.4f}")

# Classification Report
print("\n📊 Classification Report:")
print(classification_report(target_test, y_test_pred))

# Confusion Matrix
print("\n🔍 Confusion Matrix:")
print(confusion_matrix(target_test, y_test_pred))

## Conclusion

After performing additional training with feature engineering models, the best resulting validation score was still my initial LGB model. It had a validation AUC score of 0.8550. We ran the test data through that and received a final test score of 0.8500. This is our top-performing model for forecasting churn. With additional processing power, we might try additional models such as XGBoost. I had removed it since the system would time out running that mod